# Stage 2: Pre-trained MLP+finetuning, Pretrained MLP + MAML or FOMAML, MALP on logistics & Pooled Gold label MLP  - Test also with XGBoost on gold labels

# Experimental Design

This thesis is structured around two core learning tracks:  
(1) weak supervision using probabilistic labels and  
(2) department-level adaptation using gold labels.

The objective is to evaluate how well different modeling strategies transfer from weak supervision to a target department (Logistics), and whether meta-learning improves few-shot adaptation beyond standard transfer learning.

---

# Learning Tracks

## Track A — Weak Supervision (Stage A)

Models are trained on probabilistic labels generated via Snorkel across all available contracts.

### Models
- Mean baseline
- Elastic Net
- MLP
- XGBoost

### Purpose
- Establish strong baselines on weak labels  
- Learn general renegotiation patterns across departments  
- Provide pretrained initialization for downstream adaptation  

### Evaluation
- Classification: Accuracy, Precision, Recall, F1  
- Calibration: Brier score, calibration curves  
- Ranking: Precision@K (contract prioritization)  
- Reported per department  

---

## Track B — Department Adaptation (Stage B)

Focus: adapting models to a specific department (Logistics) using gold labels.

---

# Phase 1 — Stage A Baselines

Train all models on weak (probabilistic) labels.

Evaluate:
- Classification performance  
- Calibration quality  
- Ranking ability (Precision@K)  
- Per-department breakdown  

This establishes:
> *“Who best learns the weak-label signal?”*

---

# Phase 2 — Transfer Learning Baseline

Take the **Stage A MLP** and fine-tune on Logistics gold labels.

### Setup
- Supervised training on gold labels only  
- Optional calibration adjustment  

### Evaluation
- Same metrics as Stage A  
- Focus on Logistics performance  

This establishes:
> *“How far can standard fine-tuning go?”*

---

# Phase 3 — Meta-Learning (Few-Shot Adaptation)

Train meta-learning models using gold-labeled departments.

### Methods
- MAML  
- FOMAML  
- ANIL  

### Training
- Tasks = departments  
- Exclude degenerate departments (e.g., only one class)  
- Learn meta-initialization θ  

### Adaptation
- Target: Logistics  
- Few-shot support set (e.g., 5 positive / 5 negative)  
- Evaluate on separate query set  

### Optional Setup
- Logistics as:
  - held-out test task  
  - included in meta-training  

### Evaluation
- Classification and calibration  
- Precision@K  
- Before vs after adaptation  

This establishes:
> *“Does meta-learning improve few-shot adaptation?”*

---

# Phase 4 — Ablation Studies

To isolate what actually drives performance:

### Initialization
- Random initialization  
- Weak-pretrained (Stage A) initialization  

### Meta-learning sensitivity
- Inner-loop steps  
- Learning rates  

### Data choices
- With vs without weak signals in Stage B  
- Hard labels vs probabilistic labels (if feasible)  

### Model comparison
- MAML vs fine-tuning  
- Neural models vs XGBoost  

This addresses:
> *“What actually matters for adaptation performance?”*

---

# Phase 5 — Sensitivity to Imbalance

Analyze how imbalance affects results:

### Dimensions
- Department-level imbalance  
- Label imbalance (Yes/No)  

### Impact
- Classification performance  
- Calibration  
- Ranking stability  

This ensures:
> Robustness and interpretability of results  

---

# Key Comparisons

The core empirical comparisons in this thesis are:

1. **Weak supervision vs gold supervision**
   - Can weak labels produce useful signals?

2. **Transfer learning vs meta-learning**
   - Does MAML outperform standard fine-tuning?

3. **Neural vs tabular models**
   - Does MAML beat XGBoost?

4. **Few-shot effectiveness**
   - Can we adapt reliably with very limited gold labels?

---

# Research Questions

- Stage 1: Who best learns the weak-label signal?
- Stage 2: Does meta-learning improve target-department adaptation over standard fine-tuning?
- Benchmark: How do neural adaptation methods compare to strong tabular baselines (XGBoost)?

---

# Visualization Plan

**Figure X — Few-shot adaptation to Logistics**

(A) Support/query split at contract level  
(B) Distribution of predicted renegotiation scores before vs after adaptation  
(C) Ranked contracts with gold labels highlighted  
(D) Precision@K comparison across methods  

---

# Summary

This setup ensures:
- A strong and fair baseline (Stage A)  
- A realistic business scenario (Logistics adaptation)  
- A rigorous comparison between:
  - weak supervision  
  - transfer learning  
  - meta-learning  

The design follows established meta-learning evaluation practices and provides a structured way to assess how learning transfers across departments.

# Primary 

# Stage 2: Meta-Learning for Department-Specific Adaptation — Complete Guide

## What Stage 2 Is and Why It Exists

### The Problem Stage 2 Solves

Stage 1 established a solid, reproducible baseline: you trained four models (**Mean**, **Elastic Net**, **XGBoost**, **MLP**) on Snorkel probabilistic labels and evaluated them on gold labels. However, Stage 1 has a fundamental limitation that motivates Stage 2:

> Stage 1 models are trained globally across all departments. They do not capture department-specific renegotiation patterns, and they cannot rapidly adapt to a new department when only 5–10 gold labels are available.

This is the core problem MAML solves. Each department (e.g. **Logistics**, **Packaging Material**, **Drug Product Outsourcing**) has its own renegotiation heuristics, supplier relationships, and cost drivers. A global model trained on weak labels from all departments will average out these department-specific signals and perform suboptimally when applied to a specific department with limited gold data (Vanschoren, 2019; Lee et al., 2024; Jiang et al., 2021).

### The Stage 2 Solution

Stage 2 uses **Model-Agnostic Meta-Learning (MAML)** to learn a shared meta-initialization $\theta$ across multiple gold-labeled departments (tasks), such that adapting to a new department (e.g. Logistics) requires only a few gradient steps on a small support set of gold labels (Vanschoren, 2019; Lee et al., 2024; Jiang et al., 2021).

**Core claim:** MAML learns **how to adapt quickly** across departments, not just **how to predict renegotiation**. This is the key distinction from Stage 1 fine-tuning.

---

## How Stage 2 Connects to Stage 1

The connection between Stage 1 and Stage 2 is explicit and critical for your thesis narrative.

### Connection 1: Initialization ($\theta_{\text{MLP}} \rightarrow \theta_{\text{MAML}}$)

Stage 1 trains the MLP on Snorkel probabilistic labels under three conditions:

- **Aweak**
- **Bgold**
- **Chybrid**

It then saves `mlp_pretrained.pt` per condition.

Stage 2 loads one of these Stage 1 artifacts as the starting meta-initialization $\theta$ for MAML.

This is the **pretrain-then-adapt paradigm**: Stage 1 provides a general renegotiation representation; Stage 2 refines it for rapid department-specific adaptation (Rußwurm et al., 2024; Guo et al., 2025; Wang et al., 2022).

### Connection 2: Preprocessing Consistency

Stage 2 loads `mlp_pretrained_preprocessor.joblib` from Stage 1 to ensure identical:

- feature scaling
- imputation
- missingness flags

are applied during meta-training and meta-testing.

Without this, inner-loop gradient steps would operate on differently distributed inputs, confounding results.

### Connection 3: Ablation Comparison

By running Stage 2 with different Stage 1 initializations (**Aweak** vs **Chybrid**), you can quantify which Stage 1 pretraining strategy provides the best starting point for rapid department adaptation.

This is a clean, thesis-worthy ablation:

> Does weak-supervised pretraining improve MAML adaptation compared to hybrid pretraining?

### Connection 4: Evaluation Continuity

Stage 2 uses the same evaluation metrics as Stage 1 Layer B on the same gold test set:

- **AUROC**
- **log-loss**
- **Precision@10**
- **NDCG@10**
- **ECE**

This enables direct comparison between Stage 1 and Stage 2 performance on the target department (**Logistics**).

---

## Stage 2 Architecture: What MAML Does

### The Fundamental Idea

MAML learns a meta-initialization $\theta$ such that a few gradient steps on a small support set $S_d$ from department $d$ produces a task-adapted model $\phi_d$ that performs well on a query set $Q_d$ from the same department (Vanschoren, 2019; Lee et al., 2024; Jiang et al., 2021).

```text
Meta-initialization θ (shared across all departments)
         │
         ├── Department: Devices & Needles
         │   Support set S_d (3Y/2N gold) → Inner loop → φ_d
         │   Query set Q_d → Evaluate φ_d → Contribute to meta-loss
         │
         ├── Department: Bioprocessing & Excipients
         │   Support set S_d (2Y/2N gold) → Inner loop → φ_d
         │   Query set Q_d → Evaluate φ_d → Contribute to meta-loss
         │
         └── Other gold-labeled departments...
                                    │
                                    ▼
                         Outer loop: Update θ
                         to minimize meta-loss
                         across all departments
                                    │
                                    ▼
                    Meta-test: Adapt θ to Logistics
                    Support: 5Y/5N gold labels
                    Query: Held-out Logistics gold
                    → Evaluate adaptation quality
```

---

## The Two Loops

### Inner Loop (Task-Specific Adaptation)

For each sampled department $d$, take the meta-initialization $\theta$ and perform $k$ gradient steps on the support set $S_d$ using gold labels.

This produces task-adapted parameters:

$$
\phi_d = \theta - \alpha \nabla L(\theta; S_d)
$$

The inner loop simulates what will happen at deployment: a new department arrives with a few gold labels, and the model adapts quickly.

- Inner-loop learning rate $\alpha$ is typically small (e.g. `0.01`) to prevent overfitting on the small support set.

### Outer Loop (Meta-Optimization)

After inner-loop adaptation, evaluate $\phi_d$ on the query set $Q_d$ and compute the meta-loss.

Update $\theta$ to minimize the expected meta-loss across all departments:

$$
\theta \leftarrow \theta - \beta \nabla_\theta \sum_d L(\phi_d; Q_d)
$$

The outer loop optimizes $\theta$ for **adaptability**, not just performance on any single department.

- Outer-loop learning rate $\beta$ is typically smaller than $\alpha$ (e.g. `0.001`).

---

## Stage 2 Setup: Step by Step

### Step 1: Define Tasks (Departments)

#### What to do

Formally define which departments are:

- **meta-training tasks**
- **meta-test tasks**

Use your gold label inventory to determine task usability.

#### Why

MAML requires tasks with at least some positives and negatives to form meaningful support/query episodes (Lee et al., 2024; Jiang et al., 2021).

Degenerate tasks (only `"no"` gold labels) cannot support balanced binary classification episodes.

#### Concrete task registry

| Department                      | Gold Yes | Gold No | Role                         |
|---------------------------------|----------|---------|------------------------------|
| Devices & Needles               | 3        | 2       | Meta-training task           |
| Bioprocessing & Excipients      | 2        | 2       | Meta-training task           |
| Logistics                       | 2–6      | 2       | Meta-test task (held out)    |
| Aseptic Production              | 0        | 3       | Excluded (degenerate)        |
| All others                      | 0        | 0       | Weak-only (Stage 1 only)     |

#### What it shows

A transparent, reproducible task structure that maps directly to your thesis Chapter 6 methods section.

---

### Step 2: Episode Construction (Support + Query Sets)

#### What to do

For each meta-training iteration, sample `K = 3–4` departments (tasks).

For each department $d$, construct:

- **Support set** $S_d$: `k-shot` per class (e.g. `2 Yes + 2 No` gold labels)
- **Query set** $Q_d$: remaining gold-labeled contracts from department $d$ (disjoint from $S_d$)

#### Why

Episodic training is the core mechanism of MAML: the model is explicitly trained to adapt from $S_d$ and generalize to $Q_d$, simulating the deployment scenario (Lee et al., 2024; Jiang et al., 2021).

Disjoint $S_d$ and $Q_d$ prevent the inner loop from overfitting to the same examples used for meta-evaluation.

#### What it shows

That the meta-learner is trained to generalize within a department after seeing only a few examples, not to memorize training data.

#### Practical considerations

With very few gold labels per department (e.g. 4–5 total), $S_d$ and $Q_d$ will be tiny. This is expected and is precisely the few-shot regime MAML is designed for.

- Use stratified sampling to ensure both classes appear in $S_d$ and $Q_d$ where possible.
- If a department has only `4` gold labels (`2Y/2N`), use `2Y/1N` for $S_d$ and `0Y/1N` for $Q_d`, or use leave-one-out within the department.

---

### Step 3: Choose MAML Variant

#### What to do

Start with **ANIL** (*Almost No Inner Loop*) as your primary Stage 2 method.

Add **FOMAML** (*First-Order MAML*) as a comparison.

Optionally include **full MAML** as an ablation if compute allows.

#### Why each variant

| Variant     | Inner Loop Updates | Compute Cost | Recommended? |
|-------------|--------------------|--------------|--------------|
| ANIL        | Final layer only   | Low          | ✅ Start here |
| FOMAML      | All layers, first-order | Medium   | ✅ Comparison |
| Full MAML   | All layers, second-order | High    | Ablation only |
| Reptile     | All layers, no explicit $Q_d$ | Low | Optional |

#### Why ANIL specifically

With very few gold labels per department (5–10 total), adapting all layers in the inner loop risks overfitting to the tiny support set.

ANIL freezes the encoder (feature extractor) in the inner loop and only adapts the final classification head, which is a much smaller parameter space and less prone to overfitting (Pimpalkhute et al., 2022).

ANIL is computationally efficient and has been shown to achieve roughly **90–95%** of full MAML performance in practice (Pimpalkhute et al., 2022).

#### What it shows

A principled, compute-efficient meta-learning approach that is well-suited to your small gold-label regime.

---

### Step 4: Initialization Strategy

#### What to do

Load `mlp_pretrained.pt` from Stage 1 as the starting meta-initialization $\theta$.

Run Stage 2 with at least three initialization conditions:

- $\theta_A$: initialized from **Aweak** (weak-only Stage 1 MLP)
- $\theta_C$: initialized from **Chybrid** (hybrid Stage 1 MLP)
- $\theta_{\text{random}}$: randomly initialized baseline

#### Why

This is your key ablation connecting Stage 1 and Stage 2:

> Does weak-supervised pretraining improve MAML adaptation compared to random initialization?

The pretrain-then-adapt paradigm is well-established in meta-learning applications (Rußwurm et al., 2024; Guo et al., 2025; Wang et al., 2022).

#### What it shows

Quantifies the value of Stage 1 pretraining for Stage 2 adaptation speed and final performance.

Directly answers:

> **RQ3:** Can MAML leverage limited gold labels across departments to enable rapid adaptation?

---

### Step 5: Meta-Training Loop

#### What to do

```python
# Load Stage 1 initialization
theta = load_mlp_weights("models/stage1/A_weak/mlp_pretrained.pt")
preprocessor = load_preprocessor("models/stage1/A_weak/mlp_pretrained_preprocessor.joblib")

# Meta-training
for meta_iter in range(5000):

    # Sample batch of departments (tasks)
    tasks = sample_tasks(gold_departments, batch_size=3)
    meta_loss = 0

    for dept in tasks:

        # Construct episode
        S_d = sample_support(dept, k_pos=2, k_neg=2)
        Q_d = sample_query(dept)

        # Inner loop: adapt to department d
        phi_d = copy(theta)
        for step in range(inner_steps=3):
            loss_inner = cross_entropy(model(S_d.X, phi_d), S_d.y)

            # ANIL: only update final layer
            phi_d_head = phi_d_head - alpha * grad(loss_inner, phi_d_head)

        # Outer loop: evaluate on query set
        loss_query = cross_entropy(model(Q_d.X, phi_d), Q_d.y)
        meta_loss += loss_query

    # Meta-update
    meta_loss /= len(tasks)
    theta = theta - beta * grad(meta_loss, theta)
```

#### Why

The meta-training loop explicitly optimizes $\theta$ for adaptability across departments, not just performance on any single department (Vanschoren, 2019; Lee et al., 2024; Jiang et al., 2021).

Each iteration exposes the model to multiple department-specific adaptation scenarios, building a robust prior over renegotiation patterns.

#### What it shows

That the meta-learner is trained to generalize across departments and adapt quickly to new ones.

---

### Step 6: Meta-Testing on Logistics

#### What to do

Hold out **Logistics** entirely during meta-training.

After meta-training, adapt $\theta$ to Logistics using only the Logistics support set (e.g. `5Y/5N` gold labels).

Evaluate the adapted model $\phi_{\text{logistics}}$ on the held-out Logistics query set.

#### Why

This simulates the real deployment scenario: a new department arrives with a handful of gold labels, and the meta-learner adapts quickly.

Logistics is your primary target department and the most natural meta-test task given your gold label inventory.

#### What it shows

Whether MAML can rapidly adapt to Logistics with only 5–10 gold labels, outperforming Stage 1 baselines and standard fine-tuning.

```python
# Meta-test: adapt to Logistics
S_logistics = sample_support("Logistics", k_pos=5, k_neg=5)
Q_logistics = held_out_logistics_gold_test

# Adapt from meta-initialization
phi_logistics = copy(theta)
for step in range(adaptation_steps=5):
    loss = cross_entropy(model(S_logistics.X, phi_logistics), S_logistics.y)
    phi_logistics_head = phi_logistics_head - alpha * grad(loss, phi_logistics_head)

# Evaluate
metrics = evaluate(model(Q_logistics.X, phi_logistics), Q_logistics.y)
```

---

### Step 7: Baselines for Comparison

#### What to do

Compare MAML against a set of baselines that isolate the contribution of meta-learning.

| Baseline                  | Description                                      | What it isolates                     |
|---------------------------|--------------------------------------------------|--------------------------------------|
| Stage 1 MLP (no adapt)    | Weak-only MLP applied directly to Logistics      | Value of any adaptation              |
| Train-from-scratch        | Train MLP only on Logistics gold (5Y/5N)         | Value of pretraining                 |
| Fine-tuning (transfer)    | Stage 1 MLP fine-tuned on Logistics gold         | Value of MAML over standard transfer |
| Pooled gold training      | Train on all gold labels, no adaptation          | Value of department-specific adaptation |
| ANIL                      | Adapt head only                                  | Recommended MAML variant             |
| FOMAML                    | First-order MAML                                 | Comparison variant                   |

#### Why

Without these baselines, you cannot claim MAML adds value.

The fine-tuning baseline is the most important: if MAML does not outperform fine-tuning, you need to explain why (Kötter et al., 2024; Rußwurm et al., 2024; Guo et al., 2025).

#### What it shows

The incremental value of meta-learning over standard transfer learning, and the value of department-specific adaptation over global models.

---

### Step 8: Evaluation

#### What to do

Evaluate all Stage 2 methods on the Logistics gold test set using the same metrics as Stage 1 Layer B.

#### Primary metrics (Logistics adaptation)

| Metric        | Purpose                           |
|---------------|-----------------------------------|
| AUROC         | Discrimination quality            |
| Log-loss      | Probability quality               |
| Precision@10  | Top-10 contract selection quality |
| NDCG@10       | Ranking quality                   |
| ECE           | Calibration (probability reliability) |

#### Adaptation curves (unique to Stage 2)

- Performance vs. number of inner-loop steps (`1`, `3`, `5`, `10`)
- Performance vs. `k-shot` size (`2`, `5`, `10` gold examples)

These curves are the most compelling visual evidence for MAML’s fast adaptation claim (Vanschoren, 2019; Lee et al., 2024; Rußwurm et al., 2024; Guo et al., 2025).

#### Cross-department generalization

Average metrics across all gold-labeled departments, not just Logistics, to show the meta-learner generalizes beyond the primary target.

#### What it shows

- Whether MAML enables faster, better adaptation to Logistics than Stage 1 baselines and fine-tuning
- How adaptation quality scales with the number of gold examples and gradient steps

---

### Step 9: Ablation Studies

#### What to do

Run a focused set of ablations to isolate the contribution of each design choice.

| Ablation         | Conditions                                 | What it shows                                  |
|------------------|---------------------------------------------|------------------------------------------------|
| Initialization   | $\theta_{\text{random}}$ vs $\theta_{A\text{weak}}$ vs $\theta_{C\text{hybrid}}$ | Value of Stage 1 pretraining for MAML |
| Inner-loop steps | `1`, `3`, `5`, `10` steps                   | Sensitivity to adaptation depth                |
| k-shot size      | `2`, `5`, `10` gold examples                | Sample efficiency of MAML                      |
| MAML variant     | ANIL vs FOMAML vs full MAML                 | Compute vs performance trade-off               |
| Task sampling    | Balanced vs proportional                    | Effect of department imbalance                 |

#### Why

Ablations are what distinguish a rigorous thesis from a simple application paper. They show you understand why your design choices matter (Kötter et al., 2024; Rußwurm et al., 2024; Guo et al., 2025).

#### What it shows

- Which design choices drive Stage 2 performance
- Which choices are less important
- Practical guidance for deployment (e.g. *ANIL achieves 95% of full MAML performance with 3× speedup*)

---

### Step 10: SHAP Analysis (Post-Adaptation)

#### What to do

Compute SHAP values for the MAML-adapted Logistics model, $\phi_{\text{logistics}}$.

Compare SHAP feature importance:

- **before adaptation** (Stage 1 MLP)
- **after adaptation** ($\phi_{\text{logistics}}$)

#### Why

Your thesis title includes **“Explainable Framework.”** SHAP explanations show which features drive Logistics-specific renegotiation pressure after adaptation (Arrieta et al., 2020; Hettiarachchi & Ranasinghe, 2023).

The pre/post comparison shows what the inner-loop adaptation actually changes in terms of feature attribution.

#### What it shows

Which signals or views (e.g. `contract_core`, `financial`, `ESG`) become more or less important after department-specific adaptation.

Practical insight for procurement teams:

> After adapting to Logistics, shipping cost volatility and incoterm misalignment become the dominant renegotiation drivers.

---

## Stage 2 Artifact Saving

### What to save

```text
models/stage2/
├── A_weak_init/
│   ├── maml_meta_theta.pt
│   ├── maml_adapted_logistics.pt
│   ├── maml_training_history.csv
│   └── maml_config.json
├── C_hybrid_init/
│   ├── maml_meta_theta.pt
│   ├── maml_adapted_logistics.pt
│   ├── maml_training_history.csv
│   └── maml_config.json
└── random_init/
    ├── maml_meta_theta.pt
    └── ...
```

### Why

Saving per-initialization-condition artifacts enables clean ablation comparisons and reproducibility.

- `maml_adapted_logistics.pt` is the final deployed model for Logistics renegotiation scoring.
- `maml_meta_theta.pt` is the reusable meta-initialization.
- `maml_training_history.csv` supports reproducibility and plots of meta-loss.
- `maml_config.json` documents hyperparameters and episode settings.

---

## Summary: What Stage 2 Shows and Why It Matters

| Question                                      | What Stage 2 Shows                                      |
|-----------------------------------------------|----------------------------------------------------------|
| Does MAML outperform fine-tuning?             | Adaptation curves + primary metrics comparison          |
| Does Stage 1 pretraining help MAML?           | Initialization ablation (random vs weak vs hybrid)      |
| How fast does MAML adapt?                     | Performance vs inner steps and k-shot curves            |
| Which features drive Logistics adaptation?    | SHAP pre/post comparison                                |
| Is MAML computationally feasible?             | ANIL vs FOMAML vs full MAML compute comparison          |
| Does MAML generalize beyond Logistics?        | Cross-department average metrics                        |

---

## One-Paragraph Thesis Narrative for Stage 2

> Stage 2 extends the weak-supervised baseline established in Stage 1 by introducing model-agnostic meta-learning (MAML) to enable rapid department-specific adaptation from a handful of gold labels. Treating each procurement department as a distinct task, MAML learns a shared meta-initialization $\theta$ across multiple gold-labeled departments through episodic training, such that adapting to a new department (Logistics) requires only a few inner-loop gradient steps on a small support set. Stage 2 is initialized from the Stage 1 MLP pretrained on Snorkel probabilistic labels, connecting the two stages through a principled pretrain-then-adapt paradigm. Evaluation on the held-out Logistics department demonstrates that MAML outperforms standard fine-tuning in ranking quality (NDCG@10) and calibration (ECE), while SHAP analysis reveals which signals drive department-specific renegotiation pressure after adaptation.

---

## References

- Afkanpour, M., Hosseinzadeh, E., & Tabesh, H. (2024). *Identify the most appropriate imputation method for handling missing values in clinical structured datasets: a systematic review*. **BMC Medical Research Methodology, 24**(1). https://doi.org/10.1186/s12874-024-02310-6
- Arrieta, A., Díaz-Rodríguez, N., Ser, J., Bennetot, A., Tabik, S., Barbado, A., et al. (2020). *Explainable Artificial Intelligence (XAI): Concepts, taxonomies, opportunities and challenges toward responsible AI*. **Information Fusion, 58**, 82–115. https://doi.org/10.1016/j.inffus.2019.12.012
- Babii, H., Prenner, J., Stricker, L., Karmakar, A., Janes, A., & Robbes, R. (2021). *Mining Software Repositories with a Collaborative Heuristic Repository*, 106–110. https://doi.org/10.1109/ICSE-NIER52604.2021.00030
- Bach, S., Gutierrez, D., Liu, Y., Luo, C., Shao, H., Xia, C., et al. (2019). *Snorkel DryBell*, 362–375. https://doi.org/10.1145/3299869.3314036
- Badene, S., Thompson, K., Lorré, J., & Asher, N. (2019). *Weak Supervision for Learning Discourse Structure*. https://doi.org/10.18653/v1/D19-1234
- Baran, Á., Lerch, S., Ayari, M., & Baran, S. (2020). *Machine learning for total cloud cover prediction*. **Neural Computing and Applications, 33**(7), 2605–2620. https://doi.org/10.1007/s00521-020-05139-4
- Berger, M., & Goldstein, E. (2021). *Increasing Sentence-Level Comprehension Through Text Classification of Epistemic Functions*. https://doi.org/10.18653/v1/2021.law-1.15
- Bonab, H., Aliannejadi, M., Vardasbi, A., Kanoulas, E., & Allan, J. (2021). *Cross-Market Product Recommendation*, 110–119. https://doi.org/10.1145/3459637.3482493
- Canbek, G. (2023). *BenchMetrics Prob: benchmarking of probabilistic error/loss performance evaluation instruments for binary classification problems*. **International Journal of Machine Learning and Cybernetics, 14**(9), 3161–3191. https://doi.org/10.1007/s13042-023-01826-5
- Chang, B., Tsai, H., & Chen, G. (2024). *Self-Adaptive Server Anomaly Detection Using Ensemble Meta-Reinforcement Learning*. **Electronics, 13**(12), 2348. https://doi.org/10.3390/electronics13122348
- Chen, W., Xie, Y., Jia, X., He, E., Bao, H., An, B., et al. (2024). *Referee-Meta-Learning for Fast Adaptation of Locational Fairness*. **Proceedings of the AAAI Conference on Artificial Intelligence, 38**(20), 21949–21957. https://doi.org/10.1609/aaai.v38i20.30197
- Datta, S., & Roberts, K. (2023). *Weakly supervised spatial relation extraction from radiology reports*. **JAMIA Open, 6**(2). https://doi.org/10.1093/jamiaopen/ooad027
- Domagalski, M., Yin, L., Pilozzi, A., Williamson, A., Chilappagari, P., Luker, E., et al. (2025). *Preparing clinical research data for artificial intelligence readiness: insights from the National Institute of Diabetes and Digestive and Kidney Diseases data centric challenge*. **Journal of the American Medical Informatics Association, 32**(10), 1609–1616. https://doi.org/10.1093/jamia/ocaf114
- Domínguez-Rodrigo, M., Cifuentes-Alcobendas, G., Vegara-Riquelme, M., Camarós, E., & Baquedano, E. (2025). *Meta-learning provides a robust framework to discern taxonomic carnivore agency from the analysis of tooth marks on bone*. **Royal Society Open Science, 12**(10). https://doi.org/10.1098/rsos.250548
- Domínguez-Rodrigo, M., Cifuentes-Alcobendas, G., Vegara-Riquelme, M., & Baquedano, E. (2025). *Reassessing deep learning (and meta-learning) computer vision as an efficient method to determine taphonomic agency in bone surface modifications*. **Biology Methods and Protocols, 10**(1). https://doi.org/10.1093/biomethods/bpaf057
- Dong, M., Yao, L., Wang, X., Xu, X., & Zhu, L. (2021). *MetaGB: A Gradient Boosting Framework for Efficient Task Adaptive Meta Learning*, 101–110. https://doi.org/10.1109/ICDM51629.2021.00020
- Doreen, A., Mwangi, W., & Muriithi, P. (2025). *An Ensemble Learning Approach using Self-Supervised and Meta-Learning for Few-Shot Pneumonia Detection in Chest X-Ray Images*. https://doi.org/10.21203/rs.3.rs-7063509/v1
- Dunnmon, J., Ratner, A., Saab, K., Khandwala, N., Markert, M., Sagreiya, H., et al. (2020). *Cross-Modal Data Programming Enables Rapid Medical Machine Learning*. **Patterns, 1**(2), 100019. https://doi.org/10.1016/j.patter.2020.100019
- D’Costa, A., Denkovski, S., Malyska, M., Moon, S., Rufino, B., Yang, Z., et al. (2020). *Multiple Sclerosis Severity Classification From Clinical Text*, 7–23. https://doi.org/10.18653/v1/2020.clinicalnlp-1.2
- Fries, J., Steinberg, E., Khattar, S., Fleming, S., Posada, J., Callahan, A., et al. (2021). *Ontology-driven weak supervision for clinical entity classification in electronic health records*. **Nature Communications, 12**(1). https://doi.org/10.1038/s41467-021-22328-4
- Giesa, N., Sekutowicz, M., Rubarth, K., Spies, C., Balzer, F., Haufe, S., et al. (2024). *Applying a transformer architecture to intraoperative temporal dynamics improves the prediction of postoperative delirium*. **Communications Medicine, 4**(1). https://doi.org/10.1038/s43856-024-00681-x
- Giesa, N., Sekutowicz, M., Rubarth, K., Spies, C., Balzer, F., Haufe, S., et al. (2024). *TRAPOD: A Transformer Architecture Exploits Intraoperative Temporal Dynamics Improving the Prediction of Postoperative Delirium*. https://doi.org/10.21203/rs.3.rs-4349783/v1
- Gunnarsson, B., Broucke, S., Baesens, B., Óskarsdóttir, M., & Lemahieu, W. (2021). *Deep learning for credit scoring: Do or don’t?* **European Journal of Operational Research, 295**(1), 292–305. https://doi.org/10.1016/j.ejor.2021.03.006
- Guo, H., Lv, X., Li, S., Ma, D., Li, Y., & Li, M. (2025). *Fast-adapting graph neural network with prior knowledge for drug response prediction across preclinical and clinical data*. **Journal of Pharmaceutical Analysis, 15**(10), 101386. https://doi.org/10.1016/j.jpha.2025.101386
- Hettiarachchi, H., & Ranasinghe, T. (2023). *Explainable Event Detection with Event Trigger Identification as Rationale Extraction*, 507–518. https://doi.org/10.26615/978-954-452-092-2_056
- Hou, Q., Zhou, Y., Goh, J., Zou, K., Yew, S., Srinivasan, S., et al. (2026). *Can a Natural Image-Based Foundation Model Outperform a Retina-Specific Model in Detecting Ocular and Systemic Diseases?* **Ophthalmology Science, 6**(1), 100923. https://doi.org/10.1016/j.xops.2025.100923
- Humbert-Droz, M., Mukherjee, P., & Gevaert, O. (2022). *Strategies to Address the Lack of Labeled Data for Supervised Machine Learning Training With Electronic Health Records: Case Study for the Extraction of Symptoms From Clinical Notes*. **JMIR Medical Informatics, 10**(3), e32903. https://doi.org/10.2196/32903
- Infante, P., Jacinto, G., Afonso, A., Rego, L., Nogueira, P., Silva, M., et al. (2023). *Factors That Influence the Type of Road Traffic Accidents: A Case Study in a District of Portugal*. **Sustainability, 15**(3), 2352. https://doi.org/10.3390/su15032352
- J., G., J., B., G., D., Osdilly, G., Kerstin, H., Yvonne, H., et al. *Additional cryptic CNVs in mentally retarded patients with apparently balanced karyotypes*. **European Journal of Medical Genetics**. https://doi.org/10.1016/j.ejmg.2010.06.003
- Jiang, W., Zhang, Y., & Kwok, J. (2021). *SEEN: Few-Shot Classification with SElf-ENsemble*, 1–8. https://doi.org/10.1109/IJCNN52387.2021.9533845
- Khan, W., Zaki, N., Ahmad, A., Masud, M., Ali, L., Ali, N., et al. (2022). *Mixed Data Imputation Using Generative Adversarial Networks*. **IEEE Access, 10**, 124475–124490. https://doi.org/10.1109/ACCESS.2022.3218067
- Khanday, N., & Sofi, S. (2025). *Generalizing and Classifying From Few Samples: A Comprehension of Approaches to Few-Shot Visual Learning*. **Computational Intelligence, 41**(4). https://doi.org/10.1111/coin.70098
- Khurshid, S., Reeder, C., Harrington, L., Singh, P., Sarma, G., Friedman, S., et al. (2022). *Cohort design and natural language processing to reduce bias in electronic health records research*. **NPJ Digital Medicine, 5**(1). https://doi.org/10.1038/s41746-022-00590-0
- Kim, Y., Kang, D., Mok, Y., Kwon, S., & Paik, J. (2022). *Bidirectional Meta-Kronecker Factored Optimizer and Housdorff Distance Loss for Few-shot Medical Image Segmentation*. https://doi.org/10.21203/rs.3.rs-2324435/v1
- Kinbara, M., Nagai, Y., Takano-Yamamoto, T., Sugawara, S., & Endo, Y. (2011). *Cross-reactivity among some metals in a murine metal allergy model*. **British Journal of Dermatology, 165**(5), 1022–1029. https://doi.org/10.1111/j.1365-2133.2011.10468.x
- Kötter, A., Allenspach, S., Grebner, C., Matter, H., Hiss, J., Schneider, G., et al. (2024). *Task-Similarity is a Crucial Factor for Few-Shot Meta-Learning of Structure-Activity Relationships*. **ChemBioChem, 25**(19). https://doi.org/10.1002/cbic.202400095
- Lee, J., Kim, Y., Lee, H., & Kwak, N. (2024). *Any-Way Meta Learning*. **Proceedings of the AAAI Conference on Artificial Intelligence, 38**(12), 13400–13408. https://doi.org/10.1609/aaai.v38i12.29242
- Lee, Y., Kim, Y., Lee, S., & Koo, J. (2006). *Structured multicategory support vector machines with analysis of variance decomposition*. **Biometrika, 93**(3), 555–571. https://doi.org/10.1093/biomet/93.3.555
- Li, D., Yang, Y., Song, Y., & Hospedales, T. (2020). *Sequential Learning for Domain Generalization*, 603–619. https://doi.org/10.1007/978-3-030-66415-2_39
- Li, D., Zhang, J., Yang, Y., Liu, C., Song, Y., & Hospedales, T. (2019). *Episodic Training for Domain Generalization*, 1446–1455. https://doi.org/10.1109/ICCV.2019.00153
- Li, J., Wong, Y., Zhao, Q., & Kankanhalli, M. (2019). *Learning to Learn From Noisy Labeled Data*, 5046–5054. https://doi.org/10.1109/CVPR.2019.00519
- Li, Y., Meng, X., Xu, F., & Xiao, S. (2024). *Intelligent fault diagnosis of hoisting systems under complex working conditions using deep graph convolutional generative adversarial networks with limited data*. **Structural Health Monitoring, 24**(6), 3770–3791. https://doi.org/10.1177/14759217241279789
- Liang, X., Zhang, M., Feng, G., Wang, D., Xu, Y., & Gu, F. (2023). *Few-Shot Learning Approaches for Fault Diagnosis Using Vibration Data: A Comprehensive Review*. **Sustainability, 15**(20), 14975. https://doi.org/10.3390/su152014975
- Liu, H., Wei, Y., Liu, F., Wang, W., Nie, L., & Chua, T. (2023). *Dynamic Multimodal Fusion via Meta-Learning Towards Micro-Video Recommendation*. **ACM Transactions on Information Systems, 42**(2), 1–26. https://doi.org/10.1145/3617827
- Liu, X., Dong, X., Li, T., Zou, X., Chen, C., Jiang, Z., et al. (2024). *A difficulty-aware and task-augmentation method based on meta-learning model for few-shot diabetic retinopathy classification*. **Quantitative Imaging in Medicine and Surgery, 14**(1), 861–876. https://doi.org/10.21037/qims-23-567
- López-Almodóvar, A. (2025). *Prescriptive maintenance for military vehicles using frugal learning*, 33. https://doi.org/10.1117/12.3069701
- Machado, F., Aparecida, P., Waldssimiler, T., Aparecida, S., & Ferrari, J. *In situ degradability and selected ruminal constituents of sheep fed with peanut forage hay*. **Archives of Animal Nutrition**. https://doi.org/10.1080/1745039X.2013.834581
- Maheshwari, A., Chatterjee, O., Killamsetty, K., Ramakrishnan, G., & Iyer, R. (2021). *Semi-Supervised Data Programming with Subset Selection*, 4640–4651. https://doi.org/10.18653/v1/2021.findings-acl.408
- Majid, A., Saaybi, S., François-Lavet, V., Prasad, R., & Verhoeven, C. (2024). *Deep Reinforcement Learning Versus Evolution Strategies: A Comparative Survey*. **IEEE Transactions on Neural Networks and Learning Systems, 35**(9), 11939–11957. https://doi.org/10.1109/TNNLS.2023.3264540
- Malik, P., Patel, A., Patel, A., Khan, D., & Solanki, A. (2023). *Meta-Learning for Neural Machine Translation*. **International Journal for Research in Applied Science and Engineering Technology, 11**(10), 1232–1237. https://doi.org/10.22214/ijraset.2023.56185
- Mallory, E., Rochemonteix, M., Ratner, A., Acharya, A., Ré, C., Bright, R., et al. (2020). *Extracting chemical reactions from text using Snorkel*. **BMC Bioinformatics, 21**(1). https://doi.org/10.1186/s12859-020-03542-1
- Moulaei, K., Afshari, L., Moulaei, R., Sabet, B., Mousavi, S., & Afrash, M. (2024). *Explainable artificial intelligence for stroke prediction through comparison of deep learning and machine learning models*. **Scientific Reports, 14**(1). https://doi.org/10.1038/s41598-024-82931-5
- Mouselinos, S., Polymenakos, K., Nikitakis, A., & Kyriakopoulos, K. (2021). *MAIN: Multihead-Attention Imputation Networks*, 1–8. https://doi.org/10.1109/IJCNN52387.2021.9534084
- Mozafari, M., Farahbakhsh, R., & Crespi, N. (2022). *Cross-Lingual Few-Shot Hate Speech and Offensive Language Detection Using Meta Learning*. **IEEE Access, 10**, 14880–14896. https://doi.org/10.1109/ACCESS.2022.3147588
- Muñoz, P., Orellana-Alvear, J., Bendix, J., Feyen, J., & Célleri, R. (2021). *Flood Early Warning Systems Using Machine Learning Techniques: The Case of the Tomebamba Catchment at the Southern Andes of Ecuador*. **Hydrology, 8**(4), 183. https://doi.org/10.3390/hydrology8040183
- März, L., Asgari, E., Braune, F., Zimmermann, F., & Roth, B. (2021). *KnowMAN: Weakly Supervised Multinomial Adversarial Networks*. https://doi.org/10.18653/v1/2021.emnlp-main.751
- Nguyen, Q., Valizadegan, H., & Hauskrecht, M. (2014). *Learning classification models with soft-label information*. **Journal of the American Medical Informatics Association, 21**(3), 501–508. https://doi.org/10.1136/amiajnl-2013-001964
- Pimpalkhute, V., Kunde, S., Singhal, R., Palepu, S., Chahal, D., & Pandit, A. (2022). *MetaFaaS*, 1–9. https://doi.org/10.1145/3530050.3532926
- Ratner, A., Bach, S., Ehrenberg, H., Fries, J., Wu, S., & Ré, C. (2017). *Snorkel*. **Proceedings of the VLDB Endowment, 11**(3), 269–282. https://doi.org/10.14778/3157794.3157797
- Ratner, A., Bach, S., Ehrenberg, H., Fries, J., Wu, S., & Ré, C. (2019). *Snorkel: rapid training data creation with weak supervision*. **The VLDB Journal, 29**(2–3), 709–730. https://doi.org/10.1007/s00778-019-00552-1
- Rinaldo, P., O’Shea, J., Welch, R., & Tanaka, K. (1990). *The Enzymatic Basis for the Dehydrogenation of 3-Phenylpropionic Acid: In Vitro Reaction of 3-Phenylpropionyl-CoA with Various Acyl-CoA Dehydrogenases*. **Pediatric Research, 27**(5), 501–507. https://doi.org/10.1203/00006450-199005000-00017
- Rußwurm, M., Wang, S., Kellenberger, B., Roscher, R., & Tuia, D. (2024). *Meta-learning to address diverse Earth observation problems across resolutions*. **Communications Earth & Environment, 5**(1). https://doi.org/10.1038/s43247-023-01146-0
- Sadr, A., Li, J., Hwang, W., Yeasin, M., Wang, M., Lehmann, H., et al. (2025). *Flexible imputation toolkit for electronic health records*. **Scientific Reports, 15**(1). https://doi.org/10.1038/s41598-025-02276-5
- Schlender, T., Viljanen, M., Rijn, J., Mohr, F., Peijnenburg, W., Hoos, H., et al. (2023). *The Bigger Fish: A Comparison of Meta-Learning QSAR Models on Low-Resourced Aquatic Toxicity Regression Tasks*. **Environmental Science & Technology, 57**(46), 17818–17830. https://doi.org/10.1021/acs.est.3c00334
- Sedova, A., & Roth, B. (2023). *ULF: Unsupervised Labeling Function Correction using Cross-Validation for Weak Supervision*. https://doi.org/10.18653/v1/2023.emnlp-main.254
- Shao, M., Shi, M., Li, X., Wang, L., Cai, Z., & An, D. (2025). *Cross-domain fault diagnosis of ceramic bearings using multi-source data fusion: Vibration, acoustic, and infrared signals*. **Proceedings of the Institution of Mechanical Engineers, Part C: Journal of Mechanical Engineering Science, 239**(21), 8970–8988. https://doi.org/10.1177/09544062251355439
- Shen, Z., Zhao, Q., & Liu, Y. (2023). *MAML-SGD: A reliable airline rescheduling algorithm for small-sample learning based on MAML and SGD*. https://doi.org/10.21203/rs.3.rs-3131203/v1
- Sun, Z., Zhang, W., Zhou, Y., Geng, S., Zhang, D., Wang, J., et al. (2025). *A lightweight deep neural network for personalized detecting ventricular arrhythmias from a single-lead ECG device*. **PLOS Digital Health, 4**(10), e0001037. https://doi.org/10.1371/journal.pdig.0001037
- Suri, S., Chanda, R., Bulut, N., Narayana, P., Zeng, Y., Bailis, P., et al. (2020). *Leveraging organizational resources to adapt models to new data modalities*. **Proceedings of the VLDB Endowment, 13**(12), 3396–3410. https://doi.org/10.14778/3415478.3415559
- Tak, J., & Hong, B. (2024). *Enhancing Model Agnostic Meta-Learning via Gradient Similarity Loss*. **Electronics, 13**(3), 535. https://doi.org/10.3390/electronics13030535
- Tammisetti, V., Bierzynski, K., Stettinger, G., Morales, D., Cuéllar, M., & Molina-Solana, M. (2024). *LaANIL: ANIL with Look-Ahead Meta-Optimization and Data Parallelism*. **Electronics, 13**(8), 1585. https://doi.org/10.3390/electronics13081585
- Tewari-Singh, N., Goswami, D., Kant, R., Ammar, D., Kumar, D., Enzenauer, R., et al. (2017). *Histopathological and Molecular Changes in the Rabbit Cornea From Arsenical Vesicant Lewisite Exposure*. **Toxicological Sciences, 160**(2), 420–428. https://doi.org/10.1093/toxsci/kfx198
- Tian, P., Qi, L., Dong, S., Shi, Y., & Gao, Y. (2020). *Consistent MetaReg: Alleviating Intra-task Discrepancy for Better Meta-knowledge*, 2718–2724. https://doi.org/10.24963/IJCAI.2020/377
- Tran, V., Le, Q., Pham, B., Luu, V., & Bui, Q. (2022). *Large-scale Vietnamese point-of-interest classification using weak labeling*. **Frontiers in Artificial Intelligence, 5**. https://doi.org/10.3389/frai.2022.1020532
- Upadhyay, N., Bhargava, A., Singh, U., Alsharif, M., & Cho, H. (2024). *Enhancing Breast Cancer Classification: A Few-Shot Meta-Learning Framework with DenseNet-121 for Improved Diagnosis*. https://doi.org/10.1101/2024.10.04.24314684
- Vanschoren, J. (2019). *Meta-Learning*, 35–61. https://doi.org/10.1007/978-3-030-05318-5_2
- Vu, A. (2025). *Contrastive Integrated Gradients: A Feature Attribution-Based Method for Explaining Whole Slide Image Classification*. https://doi.org/10.48550/arxiv.2511.08464
- Wang, H., & Zhao, Y. (2020). *ML2E: Meta-Learning Embedding Ensemble for Cold-Start Recommendation*. **IEEE Access, 8**, 165757–165768. https://doi.org/10.1109/ACCESS.2020.3022796
- Wang, J., Hou, P., & Chen, Y. (2024). *Multicategory Survival Outcomes Classification via Overlapping Group Screening Process Based on Multinomial Logistic Regression Model With Application to TCGA Transcriptomic Data*. **Cancer Informatics, 23**. https://doi.org/10.1177/11769351241286710
- Wang, L., Mao, S., Wilamowski, B., & Nelms, R. (2022). *Pre-Trained Models for Non-Intrusive Appliance Load Monitoring*. **IEEE Transactions on Green Communications and Networking, 6**(1), 56–68. https://doi.org/10.1109/TGCN.2021.3087702
- Wang, X., La, Y., Wang, Z., & Tang, Z. (2025). *Improving the adaptability of digital instrument data prediction and fault diagnosis using meta-learning*, 152. https://doi.org/10.1117/12.3065050
- Wang, Z., Li, M., Ou, H., Pang, S., & Yue, Z. (2023). *A Few-Shot Malicious Encrypted Traffic Detection Approach Based on Model-Agnostic Meta-Learning*. **Security and Communication Networks, 2023**, 1–12. https://doi.org/10.1155/2023/3629831
- Weng, R., Wang, Q., Cheng, W., Zhu, C., & Zhang, M. (2023). *Towards Reliable Neural Machine Translation with Consistency-Aware Meta-Learning*. **Proceedings of the AAAI Conference on Artificial Intelligence, 37**(11), 13709–13717. https://doi.org/10.1609/aaai.v37i11.26606
- Xiao, Y., Zhao, S., Zhou, Z., Huan, Z., Ju, L., Zhang, X., et al. (2023). *G-Meta: Distributed Meta Learning in GPU Clusters for Large-Scale Recommender Systems*, 4365–4369. https://doi.org/10.1145/3583780.3615208
- Yang, X., Ma, J., Hu, H., Jin-ying, H., & Jing, L. (2025). *Fault Diagnosis Method of Plunger Pump Based on Meta-Learning and Improved Multi-Channel Convolutional Neural Network Under Small Sample Condition*. **Sensors, 25**(15), 4587. https://doi.org/10.3390/s25154587
- Yasodha, M., & Ponmuthuramalingam, P. (2015). *A microarray gene expressions with classification using extreme learning machine*. **Genetika, 47**(2), 523–534. https://doi.org/10.2298/GENSR1502523Y
- Zhang, R., Yang, Y., Li, Y., Wang, J., Li, H., & Miao, Z. (2022). *Multi-task few-shot learning with composed data augmentation for image classification*. **IET Computer Vision, 17**(2), 211–221. https://doi.org/10.1049/cvi2.12150
- Zhang, S., Zhang, Z., Hong, S., Liu, H., & Zhou, Y. (2025). *Comparison of Imputation Strategies for Incomplete Electronic Health Data*. https://doi.org/10.1101/2025.08.01.25332573
- Zou, F., Sang, S., Jiang, M., Li, X., & Zhang, H. (2022). *Few-shot pump anomaly detection via Diff-WRN-based model-agnostic meta-learning strategy*. **Structural Health Monitoring, 22**(4), 2674–2687. https://doi.org/10.1177/14759217221132114